# Week 3: LIBERO — Multi-Task Manipulation with Language-Conditioned Policies

**Vizuara Robot Learning Bootcamp**

In this notebook, you will:
1. Understand the LIBERO benchmark — why it's the standard for multi-task manipulation research
2. Explore 40 language-conditioned manipulation tasks across 4 generalization axes
3. Discover what makes multi-task learning fundamentally harder than single-task
4. Train Diffusion Policy on LIBERO-Spatial (10 tasks) and measure per-task success
5. Train ACT on the same tasks and compare cross-architecture performance
6. Deep dive into VQ-BeT — a third architecture that discretizes actions with Vector Quantization
7. Analyze which tasks are easy vs hard, and why some policies fail on specific tasks
8. Understand how language conditioning enables goal-directed multi-task policies

> **Hardware**: Google Colab with T4 GPU (free tier works, but Pro with A100 is faster).
>
> **Prerequisites**: Completion of Week 1 (PushT) and Week 2 (ALOHA). You should understand DDPM, 1D U-Net, FiLM conditioning, SpatialSoftmax, ACT's VAE mechanism, and the crop_shape pitfall.

---

## Part 0: Setup & Installation

We install LeRobot with the LIBERO environment extra, which pulls in the `libero` benchmark package and `robosuite` (MuJoCo-based robot simulation).

**Note**: The LIBERO installation is heavier than PushT or ALOHA — it includes robosuite, the LIBERO task library (BDDL files, init states), and MuJoCo. First-time setup may take 5-10 minutes.

In [ ]:
# Install LeRobot with LIBERO environment support
# IMPORTANT: Install EGL for headless MuJoCo rendering on Colab (no display server)
!apt-get install -qq -y libegl1-mesa-dev libgl1-mesa-dev libgles2-mesa-dev > /dev/null 2>&1
!pip install -q "lerobot[libero]" mediapy 2>&1 | tail -5

# Set MuJoCo to use EGL (headless GPU rendering) — MUST be before any MuJoCo import
import os
os.environ["MUJOCO_GL"] = "egl"

# Clear any corrupted dataset cache if you hit video decoding errors:
# !rm -rf ~/.cache/huggingface/lerobot/HuggingFaceVLA/libero

# Verify installations
import lerobot
print(f"LeRobot version: {lerobot.__version__}")
print(f"MUJOCO_GL: {os.environ.get('MUJOCO_GL', 'not set')}")

# Check LIBERO is available
try:
    from libero.libero import benchmark
    print("LIBERO benchmark: OK")
    print(f"Available suites: {list(benchmark.get_benchmark_dict().keys())}")
except ImportError:
    print("WARNING: LIBERO not installed. Dataset exploration will work, but evaluation requires LIBERO.")
    print("Run: pip install lerobot[libero]")

In [ ]:
# Core imports
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from IPython.display import HTML, display
import mediapy as media
import time
from collections import defaultdict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## Part 1: Meet LIBERO — The Multi-Task Manipulation Benchmark

### Why LIBERO?

LIBERO (Lifelong Robot Learning Benchmark) was introduced by [Liu et al. (2023)](https://libero-project.github.io/) at UT Austin. It has become **the standard benchmark** for evaluating multi-task manipulation policies in 2024-2026, used by virtually every Vision-Language-Action (VLA) paper.

**What makes LIBERO special:**
- **130+ language-conditioned tasks** — each described by natural language (e.g., *"pick up the red mug and place it to the left of the plate"*)
- **Controlled generalization axes** — 5 task suites that isolate different types of generalization
- **Standardized evaluation protocol** — per-task success rates, not just averages
- **Realistic manipulation** — Franka Panda 7-DOF robot in robosuite/MuJoCo
- **Dual-camera setup** — overhead (agentview) + wrist camera

### The Five Task Suites

LIBERO organizes tasks into 5 suites, each testing a different type of generalization:

| Suite | # Tasks | What Varies | What's Fixed | Generalization Test |
|-------|---------|-------------|-------------|--------------------|
| **LIBERO-Spatial** | 10 | Spatial arrangement of objects | Same objects, same goals | Can the policy handle objects in different positions? |
| **LIBERO-Object** | 10 | Which objects are present | Same spatial layout, same goals | Can the policy handle novel objects? |
| **LIBERO-Goal** | 10 | The goal/instruction | Same objects, same layout | Can the policy follow different instructions in the same scene? |
| **LIBERO-90** | 90 | Everything (objects, layout, goals) | — | Full diversity: the stress test |
| **LIBERO-Long** | 10 | Task horizon (long sequences) | — | Can the policy handle long-horizon tasks? |

### The Robot: Franka Panda

```
ROBOT: Franka Emika Panda (7-DOF)
  - 7 revolute joints
  - Parallel-jaw gripper
  - Impedance control in task space

OBSERVATION:
  observation.images.image   : agentview camera  [256, 256, 3]
  observation.images.image2  : wrist camera       [256, 256, 3]
  observation.state          : robot state         [8]
                               = EEF position (3) + axis-angle rotation (3) + gripper (2)

ACTION: [7] relative Cartesian control
  [dx, dy, dz, rx, ry, rz, gripper]  ∈ Box(-1, 1)
  - Translation: relative end-effector displacement
  - Rotation: relative orientation change
  - Gripper: -1 = close, +1 = open
```

### Progression from Week 1 → Week 2 → Week 3

| | Week 1: PushT | Week 2: ALOHA | Week 3: LIBERO |
|---|-------|-------|--------|
| **Action dim** | 2 (x, y) | 14 (7×2 arms) | 7 (Cartesian) |
| **Image size** | 96×96 | 480×640 | 256×256 |
| **# Cameras** | 1 | 1 | **2** (agentview + wrist) |
| **State dim** | 2 | 14 | 8 |
| **# Tasks** | 1 | 2 | **10-90** |
| **# Demos** | 205 | 50 | ~50 per task |
| **FPS** | 10 | 50 | 10 |
| **New challenge** | Learn from demos | Bimanual coordination | **Multi-task generalization** |
| **Key question** | Can diffusion generate actions? | Can it coordinate 2 arms? | **Can one policy handle many tasks?** |

---
## Part 2: Exploring the LIBERO Dataset

The pre-recorded LIBERO dataset is available on HuggingFace at `HuggingFaceVLA/libero`. It contains demonstrations across 40 tasks from the Spatial, Object, Goal, and Long suites.

Let's explore its structure, understand the multi-task organization, and visualize what makes each task unique.

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata

# Load metadata first (lightweight — no video/image download)
meta = LeRobotDatasetMetadata("HuggingFaceVLA/libero")

print("=== LIBERO Dataset Overview ===")
print(f"  FPS: {meta.fps}")
print(f"  Total episodes: {meta.total_episodes}")
print(f"  Total frames: {meta.total_frames}")
print(f"  Avg episode length: {meta.total_frames / meta.total_episodes:.0f} frames "
      f"({meta.total_frames / meta.total_episodes / meta.fps:.1f}s)")

print(f"\n=== Features ===")
for name, feat in meta.features.items():
    print(f"  {name}: {feat}")

print(f"\n=== Stats Keys ===")
for key in meta.stats.keys():
    print(f"  {key}")

In [ ]:
# Explore the task structure — LIBERO is a MULTI-TASK dataset
# meta.tasks is a pandas DataFrame, meta.episodes is an HF Dataset
# We can build the full task-to-episode mapping from metadata alone (no downloads!)

print("=== Tasks (from metadata) ===")
print(f"Tasks DataFrame columns: {list(meta.tasks.columns)}")
print(f"Number of tasks: {len(meta.tasks)}\n")

for idx, row in meta.tasks.iterrows():
    print(f"  Task {row['task_index']:2d}: {idx}")

print(f"\n=== Episodes (from metadata) ===")
print(f"Number of episodes: {len(meta.episodes)}")
print(f"Episode columns: {meta.episodes.column_names}")

# Show first few episodes
for i in range(min(5, len(meta.episodes))):
    ep = meta.episodes[i]
    print(f"  Episode {i}: tasks={ep.get('tasks', 'N/A')}, "
          f"length={ep.get('length', ep.get('dataset_to_index', 0) - ep.get('dataset_from_index', 0))}")

In [ ]:
# Load a small subset of the dataset to explore sample structure
# We load without delta_timestamps first to see raw sample format

dataset_explore = LeRobotDataset("HuggingFaceVLA/libero", episodes=[0])

print(f"Episode 0 has {len(dataset_explore)} frames")

sample = dataset_explore[0]
print(f"\n=== Sample Structure ===")
for key, val in sorted(sample.items()):
    if isinstance(val, torch.Tensor):
        print(f"  {key:35s} shape={str(val.shape):15s} dtype={val.dtype}")
    else:
        print(f"  {key:35s} = {val}")

# Check if there's a task index or language instruction
if 'task_index' in sample:
    print(f"\nTask index for episode 0: {sample['task_index']}")
if 'language_instruction' in sample:
    print(f"Language instruction: {sample['language_instruction']}")

In [ ]:
# Build task-to-episode mapping from METADATA (no episode downloads needed!)
# This is instant because we already have meta.episodes and meta.tasks loaded.

task_episodes = defaultdict(list)
task_descriptions = {}

# Build task index -> task description mapping from meta.tasks
for task_name, row in meta.tasks.iterrows():
    task_descriptions[row['task_index']] = str(task_name)

# Build task -> episode mapping from meta.episodes
for ep_idx in range(len(meta.episodes)):
    ep = meta.episodes[ep_idx]
    # The 'tasks' column contains the task description(s) for this episode
    ep_tasks = ep.get('tasks', [])
    if isinstance(ep_tasks, list) and len(ep_tasks) > 0:
        task_str = ep_tasks[0]  # Primary task
        # Find corresponding task_index
        if task_str in meta.tasks.index:
            tidx = int(meta.tasks.loc[task_str, 'task_index'])
        else:
            tidx = ep_idx  # Fallback
        task_episodes[tidx].append(ep_idx)
    elif 'task_index' in ep:
        tidx = ep['task_index']
        task_episodes[tidx].append(ep_idx)

print(f"Built task-to-episode mapping from metadata (no downloads!)")
print(f"Found {len(task_episodes)} distinct tasks\n")

for task_idx in sorted(task_episodes.keys()):
    desc = task_descriptions.get(task_idx, "(description unavailable)")
    eps = task_episodes[task_idx]
    print(f"  Task {task_idx:2d}: {len(eps):3d} episodes | {desc}")

# Total episodes mapped
total_mapped = sum(len(eps) for eps in task_episodes.values())
print(f"\nTotal episodes mapped: {total_mapped} / {len(meta.episodes)}")

In [ ]:
# Visualize frames from different tasks side by side
# This shows the visual diversity of the LIBERO benchmark

# Pick one episode from each of the first 8 tasks
tasks_to_show = sorted(task_episodes.keys())[:8]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, task_idx in enumerate(tasks_to_show):
    ep_id = task_episodes[task_idx][0]  # First episode of this task
    ds_ep = LeRobotDataset("HuggingFaceVLA/libero", episodes=[ep_id])

    # Get a frame from the middle of the episode
    mid_idx = len(ds_ep) // 2
    s = ds_ep[mid_idx]

    # Try different possible image keys
    img = None
    for img_key in ['observation.images.image', 'observation.image']:
        if img_key in s:
            img = s[img_key]
            break

    if img is not None:
        if img.dim() == 4:  # [T, C, H, W]
            img = img[-1]   # Take last frame
        img_np = img.permute(1, 2, 0).numpy()
        if img_np.max() > 1.0:
            img_np = img_np / 255.0
        img_np = np.clip(img_np, 0, 1)
        axes[i].imshow(img_np)
    else:
        axes[i].text(0.5, 0.5, 'No image', ha='center', va='center')

    desc = task_descriptions.get(task_idx, f"Task {task_idx}")
    # Wrap long descriptions
    desc_short = desc[:50] + '...' if len(desc) > 50 else desc
    axes[i].set_title(f'Task {task_idx}\n{desc_short}', fontsize=9)
    axes[i].axis('off')

plt.suptitle('LIBERO Multi-Task Diversity — One Frame per Task (agentview camera)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Each task has a different scene configuration, object set, or goal.")
print("A multi-task policy must learn to handle ALL of these from a shared set of weights.")

In [ ]:
# Visualize BOTH cameras for a single task — agentview + wrist
# LIBERO's dual-camera setup provides complementary viewpoints

ep_id = task_episodes[sorted(task_episodes.keys())[0]][0]
ds_dual = LeRobotDataset("HuggingFaceVLA/libero", episodes=[ep_id])

# Get 4 frames spread across the episode
frame_indices = np.linspace(0, len(ds_dual) - 1, 4, dtype=int)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for col, idx in enumerate(frame_indices):
    s = ds_dual[idx]
    t_sec = idx / meta.fps

    # Main camera (agentview)
    for img_key in ['observation.images.image', 'observation.image']:
        if img_key in s:
            img1 = s[img_key]
            if img1.dim() == 4: img1 = img1[-1]
            img1 = img1.permute(1, 2, 0).numpy()
            if img1.max() > 1.0: img1 = img1 / 255.0
            axes[0, col].imshow(np.clip(img1, 0, 1))
            break
    axes[0, col].set_title(f't={t_sec:.1f}s', fontsize=11)
    axes[0, col].axis('off')
    if col == 0:
        axes[0, col].set_ylabel('Agentview\n(overhead)', fontsize=12)

    # Wrist camera
    for img_key2 in ['observation.images.image2', 'observation.image2']:
        if img_key2 in s:
            img2 = s[img_key2]
            if img2.dim() == 4: img2 = img2[-1]
            img2 = img2.permute(1, 2, 0).numpy()
            if img2.max() > 1.0: img2 = img2 / 255.0
            axes[1, col].imshow(np.clip(img2, 0, 1))
            break
    axes[1, col].axis('off')
    if col == 0:
        axes[1, col].set_ylabel('Wrist\n(eye-in-hand)', fontsize=12)

plt.suptitle('Dual Camera Setup: Agentview (global context) + Wrist (fine detail)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("The agentview camera provides global scene context (where are objects?).")
print("The wrist camera provides fine-grained detail for precise manipulation.")
print("Together they give the policy both strategic and tactical information.")

In [ ]:
# Analyze the 8D state and 7D action spaces

# Collect states and actions from a few episodes
all_states = []
all_actions = []
ep_task_labels = []

for task_idx in sorted(task_episodes.keys())[:4]:  # First 4 tasks
    ep_id = task_episodes[task_idx][0]
    ds_ep = LeRobotDataset("HuggingFaceVLA/libero", episodes=[ep_id])
    for i in range(len(ds_ep)):
        s = ds_ep[i]
        if 'observation.state' in s:
            all_states.append(s['observation.state'].numpy())
        if 'action' in s:
            all_actions.append(s['action'].numpy())
            ep_task_labels.append(task_idx)

all_states = np.array(all_states) if all_states else None
all_actions = np.array(all_actions) if all_actions else None

if all_states is not None:
    print(f"States shape: {all_states.shape}")
    state_names = ['EEF_x', 'EEF_y', 'EEF_z', 'Rot_ax1', 'Rot_ax2', 'Rot_ax3', 'Grip_L', 'Grip_R']
    print(f"\nState statistics:")
    for i, name in enumerate(state_names[:all_states.shape[-1]]):
        print(f"  {name:10s}: mean={all_states[:, i].mean():.3f}  "
              f"std={all_states[:, i].std():.3f}  "
              f"range=[{all_states[:, i].min():.3f}, {all_states[:, i].max():.3f}]")

if all_actions is not None:
    print(f"\nActions shape: {all_actions.shape}")
    action_names = ['dx', 'dy', 'dz', 'rx', 'ry', 'rz', 'gripper']
    print(f"\nAction statistics:")
    for i, name in enumerate(action_names[:all_actions.shape[-1]]):
        print(f"  {name:10s}: mean={all_actions[:, i].mean():.4f}  "
              f"std={all_actions[:, i].std():.4f}  "
              f"range=[{all_actions[:, i].min():.3f}, {all_actions[:, i].max():.3f}]")

In [ ]:
# Compare action distributions across tasks
# Different tasks require different types of motions

if all_actions is not None:
    action_names = ['dx', 'dy', 'dz', 'rx', 'ry', 'rz', 'gripper']
    unique_tasks = sorted(set(ep_task_labels))
    colors = plt.cm.Set2(np.linspace(0, 1, len(unique_tasks)))

    fig, axes = plt.subplots(2, 4, figsize=(20, 8))
    axes = axes.flatten()

    for dim in range(min(7, all_actions.shape[-1])):
        for ti, task_idx in enumerate(unique_tasks):
            mask = np.array(ep_task_labels) == task_idx
            task_actions = all_actions[mask, dim]
            desc = task_descriptions.get(task_idx, f"Task {task_idx}")
            desc_short = desc[:20] + '...' if len(desc) > 20 else desc
            axes[dim].hist(task_actions, bins=50, alpha=0.5, color=colors[ti],
                          label=desc_short, density=True)
        axes[dim].set_title(action_names[dim], fontsize=12)
        axes[dim].set_xlabel('Value')
        if dim == 0:
            axes[dim].set_ylabel('Density')
        axes[dim].grid(True, alpha=0.3)

    # Legend in the last subplot
    if len(axes) > 7:
        axes[7].axis('off')
        for ti, task_idx in enumerate(unique_tasks):
            desc = task_descriptions.get(task_idx, f"Task {task_idx}")
            desc_short = desc[:35] + '...' if len(desc) > 35 else desc
            axes[7].plot([], [], color=colors[ti], linewidth=5, label=desc_short)
        axes[7].legend(loc='center', fontsize=9)

    plt.suptitle('Action Distributions by Task — Different Tasks Require Different Motions',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print("Key observations:")
    print("- Translation actions (dx, dy, dz) vary significantly across tasks")
    print("- The gripper action is bimodal: open (+1) or close (-1)")
    print("- A multi-task policy must learn to generate the RIGHT distribution for each task")

In [ ]:
# Visualize a complete episode trajectory
# Show the EEF (end-effector) path in 3D space

if all_states is not None and all_states.shape[-1] >= 3:
    fig = plt.figure(figsize=(16, 6))

    # 3D trajectory for each task
    ax3d = fig.add_subplot(121, projection='3d')
    unique_tasks = sorted(set(ep_task_labels))
    colors = plt.cm.Set2(np.linspace(0, 1, len(unique_tasks)))

    for ti, task_idx in enumerate(unique_tasks):
        mask = np.array(ep_task_labels) == task_idx
        states = all_states[mask]
        desc = task_descriptions.get(task_idx, f"Task {task_idx}")
        desc_short = desc[:25] + '...' if len(desc) > 25 else desc
        ax3d.plot(states[:, 0], states[:, 1], states[:, 2],
                 color=colors[ti], alpha=0.6, linewidth=1.5, label=desc_short)
        # Mark start
        ax3d.scatter(*states[0, :3], color=colors[ti], s=50, marker='o')

    ax3d.set_xlabel('EEF X')
    ax3d.set_ylabel('EEF Y')
    ax3d.set_zlabel('EEF Z')
    ax3d.set_title('End-Effector Trajectories (3D)', fontsize=12)
    ax3d.legend(fontsize=7, loc='upper left')

    # Gripper state over time for each task
    ax_grip = fig.add_subplot(122)
    for ti, task_idx in enumerate(unique_tasks):
        mask = np.array(ep_task_labels) == task_idx
        actions = all_actions[mask]
        t = np.arange(len(actions)) / meta.fps
        ax_grip.plot(t, actions[:, -1], color=colors[ti], alpha=0.6, linewidth=1.5)
    ax_grip.set_xlabel('Time (s)')
    ax_grip.set_ylabel('Gripper action')
    ax_grip.set_title('Gripper Commands Over Time', fontsize=12)
    ax_grip.grid(True, alpha=0.3)
    ax_grip.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax_grip.annotate('Close', xy=(0, -0.8), fontsize=10, color='gray')
    ax_grip.annotate('Open', xy=(0, 0.8), fontsize=10, color='gray')

    plt.tight_layout()
    plt.show()

    print("Each task traces a different path through workspace.")
    print("The gripper open/close pattern reveals the grasp-place structure.")

---
## Part 3: The Multi-Task Generalization Challenge

### Why is Multi-Task Harder Than Single-Task?

In Weeks 1-2, we trained a separate policy for each task (PushT, Transfer Cube). Multi-task learning asks: **can one set of neural network weights solve ALL tasks?**

This is fundamentally harder for several reasons:

**1. Conflicting Gradients**
```
Task A wants: move left to grasp mug
Task B wants: move right to grasp bowl
Same visual input (similar scene) → gradients point in opposite directions!
```
The policy must learn to condition its behavior on subtle visual or linguistic cues.

**2. Catastrophic Forgetting**
```
If we train on Task A, then Task B, then Task C...
The policy may forget how to do Task A by the time it learns Task C.
Solution: shuffle all tasks in each batch (interleaved training)
```

**3. Task Disambiguation**
- **LIBERO-Spatial**: Objects are the same but in different positions → policy must read spatial layout from images
- **LIBERO-Object**: Layout is the same but objects differ → policy must recognize objects
- **LIBERO-Goal**: EVERYTHING looks the same! → policy CANNOT distinguish tasks from vision alone — **language conditioning is essential**

**4. Capacity Bottleneck**
```
1 task  → policy memorizes one behavior
10 tasks → policy must compress 10 behaviors into shared weights
90 tasks → weights must generalize, not memorize
```

### The Language Conditioning Question

Standard Diffusion Policy and ACT (as implemented in LeRobot) **do not have language conditioning**. They rely purely on visual and proprioceptive observations to determine the task.

This works for:
- **LIBERO-Spatial** — different object positions are visually distinguishable
- **LIBERO-Object** — different objects are visually distinguishable

This **fails** for:
- **LIBERO-Goal** — same scene, different goals → visually identical!

Models with language conditioning (SmolVLA, Pi0, OpenVLA) solve this by taking the task description as additional input:
```
Standard:  policy(image, state) → action
Language:  policy(image, state, "pick up the red mug") → action
```

For this notebook, we'll train standard Diffusion Policy and ACT on **LIBERO-Spatial** where visual disambiguation is possible.

In [ ]:
# Quantify task similarity using action distributions
# High similarity between tasks = harder for the policy to distinguish them

if all_actions is not None and len(unique_tasks) >= 2:
    from itertools import combinations

    # Compute pairwise Wasserstein-1 distance between task action distributions
    # (using simple histogram overlap as a proxy)
    n_tasks = len(unique_tasks)
    similarity_matrix = np.zeros((n_tasks, n_tasks))

    for i, ti in enumerate(unique_tasks):
        for j, tj in enumerate(unique_tasks):
            mask_i = np.array(ep_task_labels) == ti
            mask_j = np.array(ep_task_labels) == tj
            actions_i = all_actions[mask_i]
            actions_j = all_actions[mask_j]
            # Simple L2 distance between mean action vectors
            dist = np.linalg.norm(actions_i.mean(axis=0) - actions_j.mean(axis=0))
            similarity_matrix[i, j] = dist

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(similarity_matrix, cmap='viridis', aspect='equal')
    ax.set_xticks(range(n_tasks))
    ax.set_yticks(range(n_tasks))
    task_labels = [f'T{t}' for t in unique_tasks]
    ax.set_xticklabels(task_labels)
    ax.set_yticklabels(task_labels)
    ax.set_title('Pairwise Action Distribution Distance Between Tasks\n'
                 '(darker = more similar = harder to distinguish)',
                 fontsize=12)
    plt.colorbar(im, label='L2 distance of mean actions')
    plt.tight_layout()
    plt.show()

    print("Tasks with similar action distributions will be harder for the policy")
    print("to distinguish without language conditioning.")

---
## Part 4: Training Diffusion Policy on LIBERO-Spatial

We'll train Diffusion Policy on the LIBERO-Spatial suite (10 tasks). Key adaptations from Week 2:

1. **Two cameras** — both agentview and wrist camera as image features
2. **crop_shape** — must match the 256x256 LIBERO images (NOT the 84x84 PushT default!)
3. **8D state** — EEF position + rotation + gripper (not 14D like ALOHA)
4. **7D action** — relative Cartesian (not joint-space like ALOHA)
5. **Multi-task data** — dataset contains episodes from multiple tasks shuffled together

**Training plan:**
- Dataset: `HuggingFaceVLA/libero` (filtered to spatial tasks if possible)
- Steps: 30,000 (use 100,000+ for best results)
- Batch size: 8 (T4 memory-friendly)
- We apply the same `make_delta_timestamps` pattern from Week 2

In [ ]:
from lerobot.policies.diffusion.configuration_diffusion import DiffusionConfig
from lerobot.policies.diffusion.modeling_diffusion import DiffusionPolicy
from lerobot.policies.factory import make_pre_post_processors
from lerobot.configs.types import FeatureType
from lerobot.datasets.utils import dataset_to_policy_features

# Helper from Week 2: convert delta indices to timestamps
def make_delta_timestamps(delta_indices, fps):
    if delta_indices is None:
        return [0.0]
    return [i / fps for i in delta_indices]

# Build policy features from dataset metadata
features = dataset_to_policy_features(meta.features)
output_features = {k: ft for k, ft in features.items() if ft.type is FeatureType.ACTION}
input_features = {k: ft for k, ft in features.items() if k not in output_features}

print("=== Input Features ===")
for k, ft in input_features.items():
    print(f"  {k}: type={ft.type}, shape={ft.shape}")

print(f"\n=== Output Features ===")
for k, ft in output_features.items():
    print(f"  {k}: type={ft.type}, shape={ft.shape}")

# Detect image size from features for crop_shape
image_features = {k: ft for k, ft in input_features.items() if ft.type is FeatureType.VISUAL}
if image_features:
    first_img_shape = next(iter(image_features.values())).shape
    img_h, img_w = first_img_shape[1], first_img_shape[2]  # C, H, W
    print(f"\nImage size: {img_h}x{img_w}")
    print(f"Number of cameras: {len(image_features)}")
    # Set crop_shape to ~87% of image size (reasonable crop that preserves most info)
    crop_h = int(img_h * 0.875) // 8 * 8  # Round to multiple of 8
    crop_w = int(img_w * 0.875) // 8 * 8
    print(f"Recommended crop_shape: ({crop_h}, {crop_w})")
else:
    crop_h, crop_w = 224, 224
    print("No image features found, using default crop_shape")

In [ ]:
# Configure Diffusion Policy for LIBERO
# CRITICAL: Set crop_shape appropriately for LIBERO's images!
# Default (84,84) would destroy the images — same lesson as Week 2.

diff_cfg = DiffusionConfig(
    input_features=input_features,
    output_features=output_features,
    # THE FIX: reasonable crop for LIBERO images
    crop_shape=(crop_h, crop_w),
    crop_is_random=True,
    n_obs_steps=2,
    horizon=16,
    n_action_steps=8,
    num_train_timesteps=100,
)

print("=== Diffusion Policy Config for LIBERO ===")
print(f"  crop_shape       = {diff_cfg.crop_shape}")
print(f"  n_obs_steps      = {diff_cfg.n_obs_steps}")
print(f"  horizon          = {diff_cfg.horizon}")
print(f"  n_action_steps   = {diff_cfg.n_action_steps}")
print(f"  num_train_ts     = {diff_cfg.num_train_timesteps}")

# Setup delta_timestamps using the config's own delta index properties
delta_timestamps_diff = {
    "action": make_delta_timestamps(diff_cfg.action_delta_indices, meta.fps),
}
delta_timestamps_diff |= {
    k: make_delta_timestamps(diff_cfg.observation_delta_indices, meta.fps)
    for k in diff_cfg.image_features
}
delta_timestamps_diff |= {
    k: make_delta_timestamps(diff_cfg.observation_delta_indices, meta.fps)
    for k in diff_cfg.state_features
}

print(f"\n=== delta_timestamps ===")
for k, v in delta_timestamps_diff.items():
    print(f"  {k}: {len(v)} timesteps — {v[:4]}{'...' if len(v) > 4 else ''}")

In [ ]:
# Load a SUBSET of the dataset for training (full dataset is too large to download)
# We select a few episodes per task to keep download manageable on Colab.
# The `episodes` parameter tells LeRobotDataset to only download those specific episodes.

EPISODES_PER_TASK = 5  # Use 5 demos per task (increase for better results)
training_episodes = []
for task_idx in sorted(task_episodes.keys()):
    eps = task_episodes[task_idx][:EPISODES_PER_TASK]
    training_episodes.extend(eps)

print(f"Training on {len(training_episodes)} episodes "
      f"({EPISODES_PER_TASK} per task × {len(task_episodes)} tasks)")
print(f"Episode indices: {training_episodes[:10]}{'...' if len(training_episodes) > 10 else ''}")

dataset_diff = LeRobotDataset(
    "HuggingFaceVLA/libero",
    episodes=training_episodes,
    delta_timestamps=delta_timestamps_diff,
)

print(f"\nMulti-task dataset size: {len(dataset_diff)} samples")

sample = dataset_diff[0]
print(f"\n=== Sample with delta_timestamps ===")
for key, val in sorted(sample.items()):
    if isinstance(val, torch.Tensor):
        print(f"  {key:35s} shape={str(val.shape):15s}")
    else:
        print(f"  {key:35s} = {val}")

In [ ]:
# Instantiate and train Diffusion Policy
diff_policy = DiffusionPolicy(diff_cfg)
diff_policy.train()
diff_policy.to(device)

diff_preprocessor, diff_postprocessor = make_pre_post_processors(
    diff_cfg, dataset_stats=meta.stats
)

n_params = sum(p.numel() for p in diff_policy.parameters())
print(f"Diffusion Policy parameters: {n_params:,} ({n_params/1e6:.1f}M)")

DIFF_TRAINING_STEPS = 30000
DIFF_BATCH_SIZE = 8
DIFF_LR = 1e-4
LOG_FREQ = 500

diff_optimizer = torch.optim.Adam(
    diff_policy.parameters(), lr=DIFF_LR,
    betas=(0.95, 0.999), eps=1e-8, weight_decay=1e-6,
)

diff_dataloader = torch.utils.data.DataLoader(
    dataset_diff, batch_size=DIFF_BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True,
)

print(f"\nTraining Diffusion Policy for {DIFF_TRAINING_STEPS} steps")
print(f"Batch size: {DIFF_BATCH_SIZE}, LR: {DIFF_LR}")
print(f"Shuffle=True ensures all tasks are interleaved — prevents catastrophic forgetting.")
print(f"Starting training...\n")

diff_losses = []
step = 0
done = False
start_time = time.time()

while not done:
    for batch in diff_dataloader:
        batch = diff_preprocessor(batch)
        loss, _ = diff_policy.forward(batch)
        loss.backward()
        diff_optimizer.step()
        diff_optimizer.zero_grad()
        diff_losses.append(loss.item())

        if step % LOG_FREQ == 0:
            elapsed = time.time() - start_time
            avg_loss = np.mean(diff_losses[-LOG_FREQ:]) if len(diff_losses) >= LOG_FREQ else np.mean(diff_losses)
            steps_per_sec = (step + 1) / elapsed
            eta = (DIFF_TRAINING_STEPS - step) / max(steps_per_sec, 0.01)
            print(f"Step {step:6d}/{DIFF_TRAINING_STEPS} | Loss: {loss.item():.4f} | "
                  f"Avg: {avg_loss:.4f} | {steps_per_sec:.1f} it/s | ETA: {eta/60:.1f}min")

        step += 1
        if step >= DIFF_TRAINING_STEPS:
            done = True
            break

total_time = time.time() - start_time
print(f"\nDiffusion training complete! {DIFF_TRAINING_STEPS} steps in {total_time/60:.1f} minutes")
print(f"Final loss: {diff_losses[-1]:.4f}")

In [ ]:
# Plot training curve and save checkpoint
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
window = min(200, len(diff_losses) // 10)

axes[0].plot(diff_losses, alpha=0.2, color='coral')
if window > 1:
    smoothed = np.convolve(diff_losses, np.ones(window)/window, mode='valid')
    axes[0].plot(range(window-1, len(diff_losses)), smoothed, 'r-', linewidth=2, label=f'Smoothed (w={window})')
axes[0].set_xlabel('Training step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Diffusion Policy Training on LIBERO (Multi-Task)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].semilogy(diff_losses, alpha=0.2, color='coral')
if window > 1:
    axes[1].semilogy(range(window-1, len(diff_losses)), smoothed, 'r-', linewidth=2)
axes[1].set_xlabel('Training step')
axes[1].set_ylabel('Loss (log scale)')
axes[1].set_title('Diffusion Loss (log scale)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

diff_output_dir = Path("outputs/diffusion_libero")
diff_output_dir.mkdir(parents=True, exist_ok=True)
diff_policy.save_pretrained(diff_output_dir)
diff_preprocessor.save_pretrained(diff_output_dir)
diff_postprocessor.save_pretrained(diff_output_dir)
print(f"Diffusion Policy saved to {diff_output_dir}")

---
## Part 5: Training ACT on LIBERO-Spatial

Now let's train ACT on the same multi-task data. Recall from Week 2:
- ACT uses a **single forward pass** (no iterative denoising)
- ACT uses a **VAE** for multimodality (not noise sampling)
- ACT's `observation_delta_indices = None` means **don't include state in delta_timestamps**

ACT handles multiple cameras natively — it concatenates the visual features from all cameras.

In [ ]:
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.policies.act.modeling_act import ACTPolicy

act_cfg = ACTConfig(
    input_features=input_features,
    output_features=output_features,
)

print("=== ACT Configuration for LIBERO ===")
print(f"  chunk_size           = {act_cfg.chunk_size}")
print(f"  n_action_steps       = {act_cfg.n_action_steps}")
print(f"  n_obs_steps          = {act_cfg.n_obs_steps}")
print(f"  dim_model            = {act_cfg.dim_model}")
print(f"  use_vae              = {act_cfg.use_vae}")
print(f"  vision_backbone      = {act_cfg.vision_backbone}")
print(f"  pretrained_weights   = {act_cfg.pretrained_backbone_weights}")

# CRITICAL: ACT delta_timestamps — NO observation.state!
delta_timestamps_act = {
    "action": make_delta_timestamps(act_cfg.action_delta_indices, meta.fps),
}
delta_timestamps_act |= {
    k: make_delta_timestamps(act_cfg.observation_delta_indices, meta.fps)
    for k in act_cfg.image_features
}

print(f"\n=== ACT delta_timestamps ===")
for k, v in delta_timestamps_act.items():
    print(f"  {k}: {len(v)} timesteps — {v[:5]}{'...' if len(v) > 5 else ''}")
print(f"\nNote: observation.state is NOT in delta_timestamps for ACT.")

In [ ]:
# Load dataset subset and train ACT (reuse same training_episodes from cell 20)
dataset_act = LeRobotDataset(
    "HuggingFaceVLA/libero",
    episodes=training_episodes,
    delta_timestamps=delta_timestamps_act,
)
print(f"ACT dataset size: {len(dataset_act)} samples")

act_policy = ACTPolicy(act_cfg)
act_policy.train()
act_policy.to(device)

act_preprocessor, act_postprocessor = make_pre_post_processors(act_cfg, dataset_stats=meta.stats)
act_params = sum(p.numel() for p in act_policy.parameters())
print(f"ACT parameters: {act_params:,} ({act_params/1e6:.1f}M)")

ACT_TRAINING_STEPS = 30000
ACT_BATCH_SIZE = 8
ACT_LR = 1e-5
LOG_FREQ = 500

act_optimizer = torch.optim.Adam(
    act_policy.parameters(), lr=ACT_LR, weight_decay=act_cfg.optimizer_weight_decay,
)
act_dataloader = torch.utils.data.DataLoader(
    dataset_act, batch_size=ACT_BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True,
)

print(f"\nTraining ACT for {ACT_TRAINING_STEPS} steps")
print(f"Starting training...\n")

act_losses = []
step = 0
done = False
start_time = time.time()

while not done:
    for batch in act_dataloader:
        batch = act_preprocessor(batch)
        loss, info = act_policy.forward(batch)
        loss.backward()
        act_optimizer.step()
        act_optimizer.zero_grad()
        act_losses.append(loss.item())

        if step % LOG_FREQ == 0:
            elapsed = time.time() - start_time
            avg_loss = np.mean(act_losses[-LOG_FREQ:]) if len(act_losses) >= LOG_FREQ else np.mean(act_losses)
            steps_per_sec = (step + 1) / elapsed
            eta = (ACT_TRAINING_STEPS - step) / max(steps_per_sec, 0.01)
            print(f"Step {step:6d}/{ACT_TRAINING_STEPS} | Loss: {loss.item():.4f} | "
                  f"Avg: {avg_loss:.4f} | {steps_per_sec:.1f} it/s | ETA: {eta/60:.1f}min")

        step += 1
        if step >= ACT_TRAINING_STEPS:
            done = True
            break

total_time = time.time() - start_time
print(f"\nACT training complete! {ACT_TRAINING_STEPS} steps in {total_time/60:.1f} minutes")

act_output_dir = Path("outputs/act_libero")
act_output_dir.mkdir(parents=True, exist_ok=True)
act_policy.save_pretrained(act_output_dir)
act_preprocessor.save_pretrained(act_output_dir)
act_postprocessor.save_pretrained(act_output_dir)
print(f"ACT saved to {act_output_dir}")

---
## Part 6: VQ-BeT — Vector-Quantized Behavior Transformers

VQ-BeT ([Lee et al., 2024](https://sjlee.cc/vq-bet/)) introduces a third approach to handling multimodal action distributions, fundamentally different from both Diffusion Policy and ACT.

### The Core Idea: Discretize Actions

Instead of predicting continuous actions directly, VQ-BeT:
1. **Learns a codebook** of action prototypes via Residual VQ-VAE
2. **Predicts which code** the action belongs to via a GPT-style transformer
3. **Predicts the offset** from the nearest code to the actual action

```
VQ-BeT = VQ-VAE (action discretization) + GPT (sequence prediction)
```

### Architecture Overview

```
PHASE 1: Train Residual VQ-VAE (first 20,000 steps)
  Action chunk [5 x 7]  -->  Encoder  -->  Quantize  -->  Decoder  -->  Reconstructed actions
                                             |
                             Primary code + Secondary code (Residual VQ)

PHASE 2: Train GPT Policy (remaining steps)
  RGB image (256x256)  --> ResNet18 --> SpatialSoftmax --> Feature tokens
  Robot state [8]      --> Linear projection -----------> State token
  Action query tokens  --> Learnable embeddings ---------> Query tokens
       |
       v
  minGPT (8 layers, 512d, 8 heads)
       |
       v
  For each action token:
    - Predict primary code (which of 16 clusters?)
    - Predict secondary code (which sub-cluster?)
    - Predict continuous offset (residual from code center)
       |
       v
  Final action = codebook[primary][secondary] + offset
```

### Why Discretize Actions?

| Approach | How it handles multimodality | Pros | Cons |
|----------|----------------------------|------|------|
| **Diffusion** | Sample from noise, denoise | Smooth, continuous | Slow (100 steps) |
| **ACT (VAE)** | Sample latent z | Fast, single pass | Limited diversity |
| **VQ-BeT** | Classify into bins + offset | Explicit modes, fast | Needs VQ-VAE pretraining |

### Two-Phase Training

1. **Phase 1 (steps 0-20000)**: Train only the Residual VQ-VAE on action chunks
2. **Phase 2 (steps 20000+)**: Train GPT policy with frozen VQ-VAE codebook

### Important Limitation

VQ-BeT in LeRobot only supports **one camera** (`validate_features` enforces `len(image_features) == 1`). For LIBERO with two cameras, we must use only the agentview camera.

In [ ]:
from lerobot.policies.vqbet.configuration_vqbet import VQBeTConfig
from lerobot.policies.vqbet.modeling_vqbet import VQBeTPolicy

# VQ-BeT only supports ONE camera — filter to agentview only
vqbet_input_features = {
    k: ft for k, ft in input_features.items()
    if not (ft.type is FeatureType.VISUAL and 'image2' in k)
}

print("=== VQ-BeT Input Features (single camera) ===")
for k, ft in vqbet_input_features.items():
    print(f"  {k}: type={ft.type}, shape={ft.shape}")

vqbet_cfg = VQBeTConfig(
    input_features=vqbet_input_features,
    output_features=output_features,
    crop_shape=(crop_h, crop_w),
    n_vqvae_training_steps=10000,
    vqvae_n_embed=16,
    vqvae_embedding_dim=256,
    gpt_n_layer=8,
    gpt_n_head=8,
    gpt_hidden_dim=512,
    n_action_pred_token=3,
    action_chunk_size=5,
    n_obs_steps=5,
)

print(f"\n=== VQ-BeT Configuration ===")
print(f"  n_obs_steps            = {vqbet_cfg.n_obs_steps}")
print(f"  n_action_pred_token    = {vqbet_cfg.n_action_pred_token}")
print(f"  action_chunk_size      = {vqbet_cfg.action_chunk_size}")
print(f"  Total actions predicted = {vqbet_cfg.n_action_pred_token * vqbet_cfg.action_chunk_size}")
print(f"  crop_shape             = {vqbet_cfg.crop_shape}")
print(f"  vqvae_n_embed          = {vqbet_cfg.vqvae_n_embed}")
print(f"  n_vqvae_training_steps = {vqbet_cfg.n_vqvae_training_steps}")

vqbet_policy = VQBeTPolicy(vqbet_cfg)
vqbet_params = sum(p.numel() for p in vqbet_policy.parameters())
print(f"\nVQ-BeT parameters: {vqbet_params:,} ({vqbet_params/1e6:.1f}M)")

print(f"\n=== Parameter Comparison ===")
print(f"  Diffusion Policy: {sum(p.numel() for p in diff_policy.parameters()):>12,}")
print(f"  ACT:              {sum(p.numel() for p in act_policy.parameters()):>12,}")
print(f"  VQ-BeT:           {vqbet_params:>12,}")

In [ ]:
# Visualize VQ-BeT action discretization concept
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

if all_actions is not None:
    from sklearn.cluster import KMeans

    axes[0].scatter(all_actions[:, 0], all_actions[:, 1], alpha=0.1, s=5, c='blue')
    axes[0].set_xlabel('dx'); axes[0].set_ylabel('dy')
    axes[0].set_title('Raw Actions (continuous)', fontsize=12)
    axes[0].grid(True, alpha=0.3)

    n_codes = 16
    actions_2d = all_actions[:, :2]
    kmeans = KMeans(n_clusters=n_codes, n_init=3, random_state=42)
    labels = kmeans.fit_predict(actions_2d)
    centers = kmeans.cluster_centers_

    for c in range(n_codes):
        mask = labels == c
        axes[1].scatter(actions_2d[mask, 0], actions_2d[mask, 1], alpha=0.15, s=5)
    axes[1].scatter(centers[:, 0], centers[:, 1], c='red', marker='x', s=200, linewidth=3,
                   label=f'{n_codes} codebook entries')
    axes[1].set_xlabel('dx'); axes[1].set_ylabel('dy')
    axes[1].set_title(f'VQ Discretization ({n_codes} codes)', fontsize=12)
    axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)

    offsets = actions_2d - centers[labels]
    axes[2].scatter(offsets[:, 0], offsets[:, 1], alpha=0.1, s=5, c='green')
    axes[2].axhline(y=0, color='red', linestyle='--', alpha=0.5)
    axes[2].axvline(x=0, color='red', linestyle='--', alpha=0.5)
    axes[2].set_xlabel('dx offset'); axes[2].set_ylabel('dy offset')
    axes[2].set_title('Offsets from Code Centers', fontsize=12)
    axes[2].grid(True, alpha=0.3)

    plt.suptitle('VQ-BeT: Action = Code (discrete) + Offset (continuous)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f"By discretizing into {n_codes} codes:")
    print(f"  - The GPT classifies actions into one of {n_codes} bins")
    print(f"  - Each bin represents a distinct action strategy")
    print(f"  - The offset head predicts the fine-grained residual")

---
## Part 7: Evaluation on LIBERO Tasks

Now let's evaluate our trained policies on the LIBERO environment. This requires the full LIBERO installation (`pip install lerobot[libero]`).

**Key evaluation details:**
- LIBERO uses `is_success` flag (binary) — the task either succeeds or fails
- We evaluate per-task success rates (not just an average)
- Each task has a maximum episode length (spatial: 280 steps)

**Important**: The LIBERO `LiberoProcessorStep` handles observation conversion:
- Images are flipped 180 degrees to match the dataset convention
- Robot state is converted: `[eef_pos(3), axis_angle(3), gripper(2)]` = 8D

In [ ]:
def evaluate_libero_policy(policy, preprocessor, postprocessor,
                           suite_name="libero_spatial", task_ids=None,
                           n_episodes_per_task=5, seed=42):
    """Evaluate a policy on LIBERO tasks. Returns per-task success rates."""
    try:
        from libero.libero import benchmark
        from lerobot.envs.libero import LiberoEnv, TASK_SUITE_MAX_STEPS
        from lerobot.processor.env_processor import LiberoProcessorStep
    except ImportError:
        print("LIBERO env not available. Install with: pip install lerobot[libero]")
        return None

    bench_dict = benchmark.get_benchmark_dict()
    suite = bench_dict[suite_name]()
    max_steps = TASK_SUITE_MAX_STEPS.get(suite_name, 300)
    if task_ids is None:
        task_ids = list(range(len(suite.tasks)))

    libero_processor = LiberoProcessorStep()
    results = {}

    for tid in task_ids:
        task_name = suite.tasks[tid].language
        print(f"\n  Task {tid}: {task_name}")
        env = LiberoEnv(
            task_suite=suite, task_id=tid, task_suite_name=suite_name,
            observation_height=256, observation_width=256, obs_type="pixels_agent_pos",
        )
        task_successes = []
        for ep in range(n_episodes_per_task):
            obs, info = env.reset(seed=seed + tid * 100 + ep)
            policy.reset()
            for step_i in range(max_steps):
                obs_dict = {}
                for cam_key, mapped_key in [('image', 'observation.images.image'),
                                            ('image2', 'observation.images.image2')]:
                    if cam_key in obs.get('pixels', {}):
                        img = obs['pixels'][cam_key]
                        img_t = torch.from_numpy(img).float().permute(2, 0, 1).unsqueeze(0) / 255.0
                        img_t = torch.flip(img_t, dims=[2, 3])  # Flip 180 to match dataset
                        obs_dict[mapped_key] = img_t.to(device)
                if 'robot_state' in obs:
                    rs = obs['robot_state']
                    eef_pos = torch.tensor(rs['eef']['pos'], dtype=torch.float32)
                    eef_quat = torch.tensor(rs['eef']['quat'], dtype=torch.float32)
                    grip = torch.tensor(rs['gripper']['qpos'], dtype=torch.float32)
                    eef_aa = libero_processor._quat2axisangle(eef_quat.unsqueeze(0)).squeeze(0)
                    state = torch.cat([eef_pos, eef_aa, grip]).unsqueeze(0).to(device)
                    obs_dict['observation.state'] = state
                obs_dict = preprocessor(obs_dict)
                with torch.no_grad():
                    action = policy.select_action(obs_dict)
                action = postprocessor(action)
                action_np = action.squeeze(0).cpu().numpy()
                obs, reward, terminated, truncated, info = env.step(action_np)
                if terminated or truncated:
                    break
            success = info.get('is_success', False)
            task_successes.append(success)
            print(f"    Ep {ep+1}: [{'OK' if success else 'FAIL'}]")
        env.close()
        sr = np.mean(task_successes) * 100
        results[tid] = {'name': task_name, 'success_rate': sr, 'successes': task_successes}
        print(f"    Success rate: {sr:.0f}%")

    avg_sr = np.mean([r['success_rate'] for r in results.values()])
    print(f"\n  Overall success rate: {avg_sr:.1f}%")
    return results

In [ ]:
# Evaluate both policies on LIBERO-Spatial
eval_task_ids = list(range(min(5, len(task_episodes))))
n_eval_episodes = 3

print("=" * 60)
print("Evaluating Diffusion Policy on LIBERO-Spatial")
print("=" * 60)
diff_policy.eval()
diff_results = evaluate_libero_policy(
    diff_policy, diff_preprocessor, diff_postprocessor,
    suite_name="libero_spatial", task_ids=eval_task_ids,
    n_episodes_per_task=n_eval_episodes,
)

print("\n" + "=" * 60)
print("Evaluating ACT on LIBERO-Spatial")
print("=" * 60)
act_policy.eval()
act_results = evaluate_libero_policy(
    act_policy, act_preprocessor, act_postprocessor,
    suite_name="libero_spatial", task_ids=eval_task_ids,
    n_episodes_per_task=n_eval_episodes,
)

In [ ]:
# Per-task success rate comparison
if diff_results and act_results:
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    task_ids_eval = sorted(diff_results.keys())
    diff_srs = [diff_results[t]['success_rate'] for t in task_ids_eval]
    act_srs = [act_results[t]['success_rate'] for t in task_ids_eval]

    x = np.arange(len(task_ids_eval))
    width = 0.35
    axes[0].bar(x - width/2, diff_srs, width, label='Diffusion', color='coral', alpha=0.8)
    axes[0].bar(x + width/2, act_srs, width, label='ACT', color='steelblue', alpha=0.8)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([f'T{t}' for t in task_ids_eval])
    axes[0].set_ylabel('Success Rate (%)')
    axes[0].set_title('Per-Task Success Rate', fontsize=13)
    axes[0].legend(); axes[0].set_ylim(0, 105); axes[0].grid(True, alpha=0.3, axis='y')

    avg_diff = np.mean(diff_srs); avg_act = np.mean(act_srs)
    bars = axes[1].bar(['Diffusion\nPolicy', 'ACT'], [avg_diff, avg_act],
                       color=['coral', 'steelblue'], width=0.5)
    axes[1].set_ylabel('Average Success Rate (%)')
    axes[1].set_title('Overall Multi-Task Success Rate', fontsize=13)
    axes[1].set_ylim(0, 105)
    for bar, val in zip(bars, [avg_diff, avg_act]):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                    f'{val:.0f}%', ha='center', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.suptitle('Diffusion vs ACT on LIBERO-Spatial (Multi-Task)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f"\n{'Task':<35s} {'Diffusion':>10s} {'ACT':>10s}")
    print("-" * 60)
    for t in task_ids_eval:
        name = diff_results[t]['name'][:33]
        print(f"{name:<35s} {diff_results[t]['success_rate']:>9.0f}% {act_results[t]['success_rate']:>9.0f}%")
    print("-" * 60)
    print(f"{'Average':<35s} {avg_diff:>9.1f}% {avg_act:>9.1f}%")
else:
    print("Evaluation results not available. Analyze training curves instead.")

---
## Part 8: Multi-Task Analysis & Discussion

In [ ]:
# Training curves comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
window = min(200, min(len(diff_losses), len(act_losses)) // 10)

if len(diff_losses) > window:
    diff_smooth = np.convolve(diff_losses, np.ones(window)/window, mode='valid')
    axes[0].plot(diff_smooth, label='Diffusion Policy', color='coral', linewidth=2)
if len(act_losses) > window:
    act_smooth = np.convolve(act_losses, np.ones(window)/window, mode='valid')
    axes[0].plot(act_smooth, label='ACT', color='steelblue', linewidth=2)
axes[0].set_xlabel('Training step'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training Curves — Multi-Task LIBERO', fontsize=13)
axes[0].legend(fontsize=11); axes[0].grid(True, alpha=0.3)

if len(diff_losses) > window:
    axes[1].semilogy(diff_smooth, label='Diffusion', color='coral', linewidth=2)
if len(act_losses) > window:
    axes[1].semilogy(act_smooth, label='ACT', color='steelblue', linewidth=2)
axes[1].set_xlabel('Training step'); axes[1].set_ylabel('Loss (log)')
axes[1].set_title('Training Curves (log scale)', fontsize=13)
axes[1].legend(fontsize=11); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Multi-task training observations:")
print("- Loss may plateau higher than single-task (the policy is doing more)")
print("- Sudden loss spikes can indicate task interference")

In [ ]:
# Summary comparison table
diff_p = sum(p.numel() for p in diff_policy.parameters())
act_p = sum(p.numel() for p in act_policy.parameters())

print("\n" + "=" * 75)
print("  THREE ARCHITECTURES ON LIBERO MULTI-TASK")
print("=" * 75)
print(f"{'Metric':<35s} {'Diffusion':>12s} {'ACT':>12s} {'VQ-BeT':>12s}")
print("-" * 75)
print(f"{'Parameters':<35s} {diff_p:>12,} {act_p:>12,} {vqbet_params:>12,}")
print(f"{'Action prediction':<35s} {'16 (exec 8)':>12s} {'100 (all)':>12s} {'15 (3x5)':>12s}")
print(f"{'Inference mechanism':<35s} {'100x denoise':>12s} {'1 forward':>12s} {'1 fwd + VQ':>12s}")
print(f"{'Multimodality':<35s} {'Noise':>12s} {'VAE latent':>12s} {'Discrete bins':>12s}")
print(f"{'Cameras supported':<35s} {'Multiple':>12s} {'Multiple':>12s} {'Single only':>12s}")
print(f"{'Language conditioning':<35s} {'No':>12s} {'No':>12s} {'No':>12s}")
print("=" * 75)

print("\n=== Published LIBERO Baselines ===")
print(f"{'Policy':<25s} {'Spatial':>8s} {'Object':>8s} {'Goal':>8s} {'Long':>8s} {'Avg':>8s}")
print("-" * 65)
print(f"{'Diffusion Policy':<25s} {'78.3%':>8s} {'92.5%':>8s} {'68.3%':>8s} {'50.5%':>8s} {'72.4%':>8s}")
print(f"{'Pi0.5 (finetuned)':<25s} {'97.0%':>8s} {'99.0%':>8s} {'98.0%':>8s} {'96.0%':>8s} {'97.5%':>8s}")
print("\nThe gap shows why language conditioning and scale matter for multi-task.")

---
## Part 9: Key Takeaways & Discussion

### What We Learned This Week

**1. Multi-task manipulation is qualitatively different from single-task**
- A single policy must handle 10+ different behaviors
- Task interference and conflicting gradients are real challenges
- Shuffled training (interleaved tasks) prevents catastrophic forgetting

**2. LIBERO's task suites isolate different generalization axes**
- Spatial: can you handle different object positions? (visual disambiguation works)
- Object: can you handle different objects? (visual disambiguation works)
- Goal: can you follow different instructions? (language conditioning required!)

**3. Standard policies (Diffusion, ACT) lack language conditioning**
- They work on Spatial and Object suites (visually distinguishable tasks)
- They fail on Goal suite (visually identical scenes, different goals)
- VLA models (Pi0, SmolVLA, OpenVLA) bridge this gap

**4. VQ-BeT offers a unique perspective on multimodality**
- Discretizes actions into codebook entries + offsets
- Explicit mode structure vs continuous noise/latent sampling
- Two-phase training adds complexity

**5. The dual-camera setup matters**
- Agentview: global context; Wrist: local detail
- Not all architectures support multiple cameras (VQ-BeT only takes one!)

**6. Per-task evaluation reveals what averages hide**
- A policy with 60% average might be 100% on easy tasks and 0% on hard ones

### Looking Ahead to Week 4: Final Projects

You now have experience with three architectures (Diffusion, ACT, VQ-BeT) and three environments (PushT, ALOHA, LIBERO). Week 4 is your chance to go deeper: ManiSkill3 for GPU-parallel simulation, 3D Diffusion Policy for point clouds, DPPO for RL fine-tuning, or cross-environment transfer.

---

## Exercises

### Exercise 1: Single-Task vs Multi-Task
Train Diffusion Policy on a **single** LIBERO-Spatial task (filter to one task's episodes). Compare success rate vs the multi-task policy.
```python
single_task_episodes = task_episodes[0]  # First task only
dataset_single = LeRobotDataset("HuggingFaceVLA/libero", episodes=single_task_episodes, ...)
```

### Exercise 2: LIBERO-Object Suite
Repeat training on LIBERO-Object. Is it harder or easier than Spatial?

### Exercise 3: LIBERO-Goal Suite (The Hard One)
Try training on LIBERO-Goal where scenes are identical but goals differ. Observe how standard policies fail without language input.

### Exercise 4: VQ-BeT Training
Train VQ-BeT end-to-end using just the agentview camera. Monitor both VQ-VAE and GPT training phases.

### Exercise 5: Scaling Analysis
Train on 1, 3, 5, 10 tasks and plot success rate. Does performance degrade linearly?

### Exercise 6 (Challenge): Language Conditioning
Research how SmolVLA or Pi0 add language conditioning. Sketch an architecture showing where the language encoder fits.

Bonus: `lerobot-eval --policy.path=lerobot/pi05_libero_finetuned --env.type=libero`

---

## Summary Table

| | Diffusion Policy | ACT | VQ-BeT |
|---|---|---|---|
| **Inference** | 100-step denoising | Single forward pass | 1 forward + VQ lookup |
| **Multimodality** | Noise initialization | VAE latent z | Discrete code bins |
| **Multi-camera** | Yes | Yes | No (single camera) |
| **Language** | No | No | No |
| **crop_shape pitfall** | Yes (default 84x84) | No | Yes (default 84x84) |
| **LIBERO-Spatial** | ~78% (published) | Good | Good |
| **LIBERO-Goal** | Poor (no language) | Poor (no language) | Poor (no language) |

**Next week**: Final projects — ManiSkill3, 3D Diffusion Policy, DPPO, or cross-environment transfer.